# LF Analysis — Phase 1 Weak Supervision

This notebook analyzes the output of the labeling functions and label model
from Phase 1 of the Neural Morphological Atomizer pipeline.

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('../src').resolve()))

## 1. Load Data

In [ ]:
candidates_path = Path('../data/prelabeled/candidates.jsonl')
labels_path = Path('../data/weak_labels/probabilistic_labels.jsonl')

candidates = [json.loads(l) for l in open(candidates_path)]
labels = [json.loads(l) for l in open(labels_path)]

print(f'Candidates: {len(candidates)} tokens')
print(f'Labels: {len(labels)} tokens')

## 2. Parse Count Distribution

In [ ]:
parse_counts = [c['parse_count'] for c in candidates]
buckets = Counter()
for pc in parse_counts:
    buckets[str(pc) if pc <= 4 else '5+'] += 1

keys = ['0', '1', '2', '3', '4', '5+']
vals = [buckets.get(k, 0) for k in keys]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(keys, vals, color='steelblue')
ax.set_xlabel('Parse Count')
ax.set_ylabel('Number of Tokens')
ax.set_title('Distribution of Candidate Parse Counts per Token')
for i, v in enumerate(vals):
    ax.text(i, v + 500, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f'\nUnambiguous (1 parse): {buckets["1"]} ({buckets["1"]/len(candidates):.1%})')
print(f'Ambiguous (2+ parses): {sum(v for k,v in buckets.items() if k != "0" and k != "1")} '
      f'({sum(v for k,v in buckets.items() if k != "0" and k != "1")/len(candidates):.1%})')

## 3. Per-LF Coverage, Overlap, and Conflict Rates

In [ ]:
from data.labeling_functions import ALL_LFS, build_label_matrix, compute_lf_stats

label_matrix, records = build_label_matrix(
    Path('../data/prelabeled/candidates.jsonl'),
    Path('../data/processed/corpus.jsonl'),
)

lf_stats = compute_lf_stats(label_matrix)

print(f'{"LF Name":25s} {"Coverage":>10s} {"Overlap":>10s} {"Conflict":>10s}')
print('-' * 57)
for name, stats in lf_stats.items():
    print(f'{name:25s} {stats["coverage"]:10.3f} {stats["overlap_rate"]:10.3f} {stats["conflict_rate"]:10.3f}')

## 4. Confidence Score Distribution

In [ ]:
confidences = [l['confidence'] for l in labels]
entropies = [l['entropy'] for l in labels]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(confidences, bins=50, color='seagreen', edgecolor='white')
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Confidence Scores')
axes[0].axvline(x=0.6, color='red', linestyle='--', label='Review threshold (0.6)')
axes[0].legend()

axes[1].hist(entropies, bins=50, color='coral', edgecolor='white')
axes[1].set_xlabel('Entropy')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Entropy Scores')
axes[1].axvline(x=1.5, color='red', linestyle='--', label='Review threshold (1.5)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Human Review Statistics

In [ ]:
needs_review = [l for l in labels if l['needs_human_review']]
no_review = [l for l in labels if not l['needs_human_review']]

print(f'Total tokens:            {len(labels)}')
print(f'Needs human review:      {len(needs_review)} ({len(needs_review)/len(labels):.1%})')
print(f'Auto-labeled (confident): {len(no_review)} ({len(no_review)/len(labels):.1%})')

## 6. Top-20 Most Ambiguous Tokens

In [ ]:
# Find tokens with highest parse counts
by_parse_count = sorted(candidates, key=lambda c: c['parse_count'], reverse=True)
seen = set()
top_ambiguous = []
for c in by_parse_count:
    if c['surface'] not in seen:
        seen.add(c['surface'])
        top_ambiguous.append(c)
        if len(top_ambiguous) >= 20:
            break

print(f'{"Rank":>4s}  {"Token":20s} {"Parse Count":>12s}')
print('-' * 40)
for i, c in enumerate(top_ambiguous, 1):
    print(f'{i:4d}  {c["surface"]:20s} {c["parse_count"]:12d}')

## 7. Zeyrek vs TRMorph Agreement

Note: TRMorph is not available on this node (requires foma). Agreement rate is 0%.
When TRMorph becomes available, re-run prelabeling with `backends=['zeyrek', 'trmorph']`.

In [ ]:
agree_count = sum(1 for c in candidates if c.get('zeyrek_trmorph_agree', False))
print(f'Zeyrek-TRMorph agreement: {agree_count}/{len(candidates)} ({agree_count/len(candidates):.1%})')
print('(TRMorph not available on this node — agreement will improve when foma is installed)')